##Name: Nawaraj Tamang

# Banking SQL Practice — PostgreSQL

40-question SQL practice set against a small banking dataset (`customers`,
`accounts`, `transactions`), covering joins, aggregation, subqueries, set
operations, CTEs/views, window functions, data-quality checks, CASE, and a
transactional UPDATE+INSERT block.

Credentials are loaded from a local `.env` file (never hard-coded), and every
operation — connect, create schema, load data, each query, close — is logged
to `log.log`.

**Setup before running:** fill in your real Postgres host/port/dbname/user/password, then
`pip install -r requirements.txt`.

## 1. Imports & Logging

In [1]:
import logging
import os


import psycopg2
from psycopg2 import IntegrityError, OperationalError, Error as Psycopg2Error
from psycopg2.extras import RealDictCursor
from dotenv import load_dotenv

logging.basicConfig(
    filename="log.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("banking_sql")
print("Logging configured -> log.log")

Logging configured -> log.log


## 2. Load credentials from `.env`

In [2]:
load_dotenv(override=True)  # looks for a ".env" file relative to this notebook

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")

_required = {"DB_HOST": DB_HOST, "DB_PORT": DB_PORT, "DB_NAME": DB_NAME,
             "DB_USER": DB_USER, "DB_PASSWORD": DB_PASSWORD}
_missing = [k for k, v in _required.items() if not v]
if _missing:
    raise RuntimeError(
        f"Missing required .env variable(s): {', '.join(_missing)}. "
        f"Please check your .env file."
    )
print("Credentials loaded from .env")

Credentials loaded from .env


## 3. Connection manager (Day 5-6 OOP style)

In [3]:
class DBConnection:
    """Small wrapper so every part of the pipeline shares one connection."""

    def __init__(self, host, port, dbname, user, password):
        self.conn = psycopg2.connect(
            host=host, port=port, dbname=dbname, user=user, password=password
        )
        logger.info("Connected to database '%s'", dbname)

    def cursor(self):
        return self.conn.cursor(cursor_factory=RealDictCursor)

    def commit(self):
        self.conn.commit()

    def rollback(self):
        self.conn.rollback()

    def close(self):
        self.conn.close()
        logger.info("Connection closed")


db = DBConnection(host=DB_HOST, port=DB_PORT, dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD)
print("Connected")

Connected


## 4. Create schema

Creates `customers`, `accounts` and `transactions` per the Instructions tab.

The source data intentionally contains a couple of "broken" rows (an
orphaned account whose `customer_id` has no matching customer, a duplicate
customer, and a few blank `city`/`email` values) so the Data Quality
questions (#2, #3, #32, #33) have something to find. Because of the orphaned
account, the FK on `accounts.customer_id` is added as `NOT VALID` *after*
the data is loaded, rather than inline in `CREATE TABLE`.

In [5]:
DDL_TABLES = """
CREATE TABLE IF NOT EXISTS customers (
    customer_id     INT PRIMARY KEY,
    first_name      VARCHAR(50),
    last_name       VARCHAR(50),
    gender          VARCHAR(10),
    dob             DATE,
    city            VARCHAR(50),
    state           VARCHAR(50),
    country         VARCHAR(50),
    phone           VARCHAR(20),
    email           VARCHAR(100),
    occupation      VARCHAR(50),
    annual_income   NUMERIC(14, 2),
    credit_score    INT,
    kyc_status      VARCHAR(20),
    join_date       DATE,
    risk_category   VARCHAR(20)
);

CREATE TABLE IF NOT EXISTS accounts (
    account_id          INT PRIMARY KEY,
    customer_id         INT,
    account_type        VARCHAR(30),
    branch               VARCHAR(50),
    ifsc_code            VARCHAR(15),
    currency             VARCHAR(5),
    balance              NUMERIC(14, 2),
    interest_rate        NUMERIC(5, 2),
    open_date            DATE,
    close_date           DATE,
    status               VARCHAR(20),
    is_joint_account     BOOLEAN
);

CREATE TABLE IF NOT EXISTS transactions (
    transaction_id   INT PRIMARY KEY,
    account_id       INT REFERENCES accounts(account_id),
    txn_date         DATE,
    txn_time         TIME,
    txn_type         VARCHAR(30),
    channel          VARCHAR(30),
    amount           NUMERIC(14, 2),
    currency         VARCHAR(5),
    balance_after    NUMERIC(14, 2),
    merchant         VARCHAR(60),
    description      VARCHAR(100),
    is_flagged       BOOLEAN
);
"""



In [6]:
print(db)

In [7]:

# Added separately, and NOT VALID, because one account row is intentionally
# orphaned (customer_id 99999 does not exist in customers).
DDL_FK_NOT_VALID = """
ALTER TABLE accounts
    DROP CONSTRAINT IF EXISTS accounts_customer_id_fkey;

ALTER TABLE accounts
    ADD CONSTRAINT accounts_customer_id_fkey
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
    NOT VALID;
"""


def create_schema(db):
    try:
        with db.cursor() as cur:
            cur.execute(DDL_TABLES)
        db.commit()
        logger.info("Schema created/verified (customers, accounts, transactions)")
        print("Schema created/verified")
    except OperationalError as e:
        db.rollback()
        logger.error("Schema creation failed: %s", e)
        raise


def add_orphan_tolerant_fk(db):
    """Adds the accounts -> customers FK as NOT VALID so the orphaned
    account row (customer_id 99999) doesn't block it."""
    try:
        with db.cursor() as cur:
            cur.execute(DDL_FK_NOT_VALID)
        db.commit()
        logger.info("Added NOT VALID FK accounts.customer_id -> customers.customer_id")
        print("FK added (NOT VALID, to tolerate the intentional orphaned row)")
    except OperationalError as e:
        db.rollback()
        logger.error("Adding FK failed: %s", e)
        raise


create_schema(db)

Schema created/verified


## 5. Load data

Loads `data/customers.csv`, `data/accounts.csv` and `data/transactions.csv`
using `COPY` (fast, and `NULL ''` converts blank CSV fields to real SQL
`NULL`s). Order matters: customers -> accounts -> transactions.

In [8]:
def truncate_all(db):
    """Wipes existing rows so this notebook is safely re-runnable."""
    with db.cursor() as cur:
        cur.execute("TRUNCATE TABLE transactions, accounts, customers RESTART IDENTITY CASCADE;")
    db.commit()
    logger.info("Truncated customers, accounts, transactions before reload")


def load_csv(db, table: str, csv_path: str):
    with db.cursor() as cur:
        with open(csv_path, "r", encoding="utf-8") as f:
            cur.copy_expert(
                f"COPY {table} FROM STDIN WITH (FORMAT csv, HEADER true, NULL '')",
                f,
            )
    db.commit()
    with db.cursor() as cur:
        cur.execute(f"SELECT COUNT(*) AS n FROM {table};")
        count = cur.fetchone()["n"]
    logger.info("Loaded %s rows into %s from %s", count, table, csv_path)
    print(f"Loaded {count} rows into {table}")


try:
    truncate_all(db)
    load_csv(db, "customers", "data/customers.csv")
    load_csv(db, "accounts", "data/accounts.csv")
    load_csv(db, "transactions", "data/transactions.csv")
    add_orphan_tolerant_fk(db)
    print("All data loaded successfully.")
    logger.info("Data load complete")
except Psycopg2Error as e:
    db.rollback()
    logger.error("Data load failed: %s", e)
    raise

Loaded 201 rows into customers
Loaded 280 rows into accounts
Loaded 4613 rows into transactions
FK added (NOT VALID, to tolerate the intentional orphaned row)
All data loaded successfully.


## 6. Query runner helper

Executes a query, logs it, prints a short preview (row count + up to 5 sample
rows), and returns the full result set as a list of dicts.

In [17]:
def run(db, label: str, query: str, params=None, preview: int = 5):
    with db.cursor() as cur:
        cur.execute(query, params)
        rows = [] if cur.description is None else cur.fetchall()
    db.commit()
    logger.info("%s -> %d row(s)", label, len(rows))
    print(f"\n{label}  ({len(rows)} row(s))")
    for r in rows[:preview]:
        print("  ", dict(r))
    if len(rows) > preview:
        print(f"   ... {len(rows) - preview} more row(s)")
    return rows

## 7. Questions 1-5 — Joins

In [16]:
q1 = run(db, "Q1: Active accounts with owner name/email", """
    SELECT a.account_id, a.balance, a.status,
           c.first_name || ' ' || c.last_name AS full_name, c.email
    FROM accounts a
    JOIN customers c ON a.customer_id = c.customer_id
    WHERE a.status = 'Active';
""")



Q1: Active accounts with owner name/email  (192 row(s))
   {'account_id': 100001, 'balance': Decimal('179640.10'), 'status': 'Active', 'full_name': 'Krishna Rai', 'email': 'krishna.rai1@mailbank.com'}
   {'account_id': 100002, 'balance': Decimal('757474.71'), 'status': 'Active', 'full_name': 'Anita Lama', 'email': 'anita.lama2@mailbank.com'}
   {'account_id': 100004, 'balance': Decimal('84465.72'), 'status': 'Active', 'full_name': 'Gita Pandey', 'email': 'gita.pandey3@mailbank.com'}
   {'account_id': 100007, 'balance': Decimal('4350.72'), 'status': 'Active', 'full_name': 'Indira Gurung', 'email': 'indira.gurung5@mailbank.com'}
   {'account_id': 100008, 'balance': Decimal('252651.41'), 'status': 'Active', 'full_name': 'Sabina Bhattarai', 'email': 'sabina.bhattarai6@mailbank.com'}
   {'account_id': 100009, 'balance': Decimal('40417.94'), 'status': 'Active', 'full_name': 'Sabina Bhattarai', 'email': 'sabina.bhattarai6@mailbank.com'}
   {'account_id': 100011, 'balance': Decimal('657401.46

In [18]:
q2 = run(db, "Q2: Customers with no account at all", """
    SELECT c.*
    FROM customers c
    LEFT JOIN accounts a ON c.customer_id = a.customer_id
    WHERE a.account_id IS NULL;
""")


Q2: Customers with no account at all  (1 row(s))
   {'customer_id': 9999, 'first_name': 'Sabina', 'last_name': 'Bhattarai', 'gender': 'Male', 'dob': datetime.date(1976, 11, 24), 'city': 'Pokhara', 'state': 'Gandaki', 'country': 'Nepal', 'phone': '9848548922', 'email': 'sabina.bhattarai6@mailbank.com', 'occupation': 'Government Employee', 'annual_income': Decimal('2291289.95'), 'credit_score': 651, 'kyc_status': 'Pending', 'join_date': datetime.date(2020, 6, 16), 'risk_category': 'High'}


In [19]:
q3 = run(db, "Q3: Orphaned accounts (no matching customer)", """
    SELECT a.*
    FROM accounts a
    LEFT JOIN customers c ON a.customer_id = c.customer_id
    WHERE c.customer_id IS NULL;
""")


Q3: Orphaned accounts (no matching customer)  (1 row(s))
   {'account_id': 100280, 'customer_id': 99999, 'account_type': 'Savings', 'branch': 'Kathmandu Main', 'ifsc_code': 'KTMN0001', 'currency': 'NPR', 'balance': Decimal('15000.00'), 'interest_rate': Decimal('4.00'), 'open_date': datetime.date(2023, 5, 10), 'close_date': None, 'status': 'Active', 'is_joint_account': False}


In [20]:
q4 = run(db, "Q4: Full outer join, labeled Matched/No Account/Missing Customer", """
    SELECT c.customer_id, a.account_id,
           CASE
               WHEN c.customer_id IS NOT NULL AND a.account_id IS NOT NULL THEN 'Matched'
               WHEN c.customer_id IS NOT NULL AND a.account_id IS NULL THEN 'No Account'
               ELSE 'Missing Customer'
           END AS match_status
    FROM customers c
    FULL OUTER JOIN accounts a ON c.customer_id = a.customer_id;
""")


Q4: Full outer join, labeled Matched/No Account/Missing Customer  (281 row(s))
   {'customer_id': 1, 'account_id': 100001, 'match_status': 'Matched'}
   {'customer_id': 2, 'account_id': 100002, 'match_status': 'Matched'}
   {'customer_id': 3, 'account_id': 100003, 'match_status': 'Matched'}
   {'customer_id': 3, 'account_id': 100004, 'match_status': 'Matched'}
   {'customer_id': 3, 'account_id': 100005, 'match_status': 'Matched'}
   ... 276 more row(s)


In [21]:
q5 = run(db, "Q5: Transaction id/amount + account type/branch + customer name (3-way join)", """
    SELECT t.transaction_id, t.amount, a.account_type, a.branch,
           c.first_name || ' ' || c.last_name AS customer_name
    FROM transactions t
    JOIN accounts a ON t.account_id = a.account_id
    JOIN customers c ON a.customer_id = c.customer_id;
""")


Q5: Transaction id/amount + account type/branch + customer name (3-way join)  (4613 row(s))
   {'transaction_id': 7000011, 'amount': Decimal('272.45'), 'account_type': 'Checking', 'branch': 'Dharan North', 'customer_name': 'Krishna Rai'}
   {'transaction_id': 7000017, 'amount': Decimal('46086.82'), 'account_type': 'Checking', 'branch': 'Dharan North', 'customer_name': 'Krishna Rai'}
   {'transaction_id': 7000012, 'amount': Decimal('9482.80'), 'account_type': 'Checking', 'branch': 'Dharan North', 'customer_name': 'Krishna Rai'}
   {'transaction_id': 7000002, 'amount': Decimal('19581.27'), 'account_type': 'Checking', 'branch': 'Dharan North', 'customer_name': 'Krishna Rai'}
   {'transaction_id': 7000005, 'amount': Decimal('62000.39'), 'account_type': 'Checking', 'branch': 'Dharan North', 'customer_name': 'Krishna Rai'}
   ... 4608 more row(s)


## 8. Questions 6-10 — Aggregation

In [22]:
q6 = run(db, "Q6: Total balance per branch, highest first", """
    SELECT branch, SUM(balance) AS total_balance
    FROM accounts
    GROUP BY branch
    ORDER BY total_balance DESC;
""")


Q6: Total balance per branch, highest first  (8 row(s))
   {'branch': 'Pokhara City', 'total_balance': Decimal('10680073.98')}
   {'branch': 'Butwal West', 'total_balance': Decimal('10638192.58')}
   {'branch': 'Itahari Plaza', 'total_balance': Decimal('10574591.36')}
   {'branch': 'Biratnagar East', 'total_balance': Decimal('8855755.92')}
   {'branch': 'Bhaktapur Central', 'total_balance': Decimal('8556332.73')}
   ... 3 more row(s)


In [23]:
q7 = run(db, "Q7: Top 5 branches by total balance (Active accounts only)", """
    SELECT branch, SUM(balance) AS total_balance
    FROM accounts
    WHERE status = 'Active'
    GROUP BY branch
    ORDER BY total_balance DESC
    LIMIT 5;
""")


Q7: Top 5 branches by total balance (Active accounts only)  (5 row(s))
   {'branch': 'Bhaktapur Central', 'total_balance': Decimal('8429354.28')}
   {'branch': 'Biratnagar East', 'total_balance': Decimal('8104185.98')}
   {'branch': 'Itahari Plaza', 'total_balance': Decimal('7743224.00')}
   {'branch': 'Butwal West', 'total_balance': Decimal('7725054.95')}
   {'branch': 'Pokhara City', 'total_balance': Decimal('7390898.09')}


In [24]:
q8 = run(db, "Q8: Account types with average balance > 50,000", """
    SELECT account_type, ROUND(AVG(balance), 2) AS avg_balance
    FROM accounts
    GROUP BY account_type
    HAVING AVG(balance) > 50000
    ORDER BY avg_balance DESC;
""")


Q8: Account types with average balance > 50,000  (4 row(s))
   {'account_type': 'Fixed Deposit', 'avg_balance': Decimal('410096.22')}
   {'account_type': 'Recurring Deposit', 'avg_balance': Decimal('356654.44')}
   {'account_type': 'Savings', 'avg_balance': Decimal('115148.34')}
   {'account_type': 'Checking', 'avg_balance': Decimal('70698.04')}


In [25]:
q9 = run(db, "Q9: Customers holding more than 1 account", """
    SELECT customer_id, COUNT(*) AS num_accounts
    FROM accounts
    GROUP BY customer_id
    HAVING COUNT(*) > 1
    ORDER BY num_accounts DESC;
""")


Q9: Customers holding more than 1 account  (59 row(s))
   {'customer_id': 162, 'num_accounts': 3}
   {'customer_id': 30, 'num_accounts': 3}
   {'customer_id': 84, 'num_accounts': 3}
   {'customer_id': 31, 'num_accounts': 3}
   {'customer_id': 115, 'num_accounts': 3}
   ... 54 more row(s)


In [26]:
q10 = run(db, "Q10: Branch + account_type combo with highest total transaction amount", """
    SELECT a.branch, a.account_type, SUM(t.amount) AS total_amount
    FROM transactions t
    JOIN accounts a ON t.account_id = a.account_id
    GROUP BY a.branch, a.account_type
    ORDER BY total_amount DESC
    LIMIT 1;
""")


Q10: Branch + account_type combo with highest total transaction amount  (1 row(s))
   {'branch': 'Butwal West', 'account_type': 'Recurring Deposit', 'total_amount': Decimal('7749589.46')}


## 9. Questions 11-16 — Subqueries

In [27]:
q11 = run(db, "Q11: Customers whose combined balance exceeds overall average", """
    SELECT c.customer_id, c.first_name, c.last_name, SUM(a.balance) AS total_balance
    FROM customers c
    JOIN accounts a ON c.customer_id = a.customer_id
    GROUP BY c.customer_id, c.first_name, c.last_name
    HAVING SUM(a.balance) > (SELECT AVG(balance) FROM accounts)
    ORDER BY total_balance DESC;
""")


Q11: Customers whose combined balance exceeds overall average  (97 row(s))
   {'customer_id': 115, 'first_name': 'Rajesh', 'last_name': 'Chaudhary', 'total_balance': Decimal('1738508.08')}
   {'customer_id': 148, 'first_name': 'Bipin', 'last_name': 'Ghimire', 'total_balance': Decimal('1562901.71')}
   {'customer_id': 26, 'first_name': 'Sita', 'last_name': 'Adhikari', 'total_balance': Decimal('1562656.70')}
   {'customer_id': 165, 'first_name': 'Anjali', 'last_name': 'Tamang', 'total_balance': Decimal('1474065.84')}
   {'customer_id': 163, 'first_name': 'Meera', 'last_name': 'Thapa', 'total_balance': Decimal('1273404.68')}
   ... 92 more row(s)


In [28]:
q12 = run(db, "Q12: Accounts above the average balance of their own account_type (correlated)", """
    SELECT a.account_id, a.account_type, a.balance
    FROM accounts a
    WHERE a.balance > (
        SELECT AVG(a2.balance) FROM accounts a2 WHERE a2.account_type = a.account_type
    )
    ORDER BY a.account_type, a.balance DESC;
""")


Q12: Accounts above the average balance of their own account_type (correlated)  (124 row(s))
   {'account_id': 100001, 'account_type': 'Checking', 'balance': Decimal('179640.10')}
   {'account_id': 100218, 'account_type': 'Checking', 'balance': Decimal('169342.75')}
   {'account_id': 100187, 'account_type': 'Checking', 'balance': Decimal('166046.36')}
   {'account_id': 100123, 'account_type': 'Checking', 'balance': Decimal('156007.95')}
   {'account_id': 100171, 'account_type': 'Checking', 'balance': Decimal('153248.69')}
   ... 119 more row(s)


In [29]:
q13 = run(db, "Q13: Customers with at least one Withdrawal transaction (EXISTS)", """
    SELECT c.*
    FROM customers c
    WHERE EXISTS (
        SELECT 1
        FROM transactions t
        JOIN accounts a ON t.account_id = a.account_id
        WHERE a.customer_id = c.customer_id AND t.txn_type = 'Withdrawal'
    );
""")


Q13: Customers with at least one Withdrawal transaction (EXISTS)  (175 row(s))
   {'customer_id': 1, 'first_name': 'Krishna', 'last_name': 'Rai', 'gender': 'Female', 'dob': datetime.date(1959, 5, 2), 'city': 'Pokhara', 'state': 'Gandaki', 'country': 'Nepal', 'phone': '9845962432', 'email': 'krishna.rai1@mailbank.com', 'occupation': 'Engineer', 'annual_income': Decimal('1650069.45'), 'credit_score': 359, 'kyc_status': 'Expired', 'join_date': datetime.date(2018, 10, 25), 'risk_category': 'Low'}
   {'customer_id': 2, 'first_name': 'Anita', 'last_name': 'Lama', 'gender': 'Female', 'dob': datetime.date(1961, 4, 8), 'city': 'Biratnagar', 'state': 'Koshi', 'country': 'Nepal', 'phone': '9816087647', 'email': 'anita.lama2@mailbank.com', 'occupation': 'Software Developer', 'annual_income': Decimal('1886567.14'), 'credit_score': 879, 'kyc_status': 'Verified', 'join_date': datetime.date(2019, 1, 3), 'risk_category': 'High'}
   {'customer_id': 3, 'first_name': 'Gita', 'last_name': 'Pandey', 'gende

In [30]:
q14 = run(db, "Q14: Accounts that have never had a transaction (NOT EXISTS)", """
    SELECT a.*
    FROM accounts a
    WHERE NOT EXISTS (
        SELECT 1 FROM transactions t WHERE t.account_id = a.account_id
    );
""")


Q14: Accounts that have never had a transaction (NOT EXISTS)  (1 row(s))
   {'account_id': 100280, 'customer_id': 99999, 'account_type': 'Savings', 'branch': 'Kathmandu Main', 'ifsc_code': 'KTMN0001', 'currency': 'NPR', 'balance': Decimal('15000.00'), 'interest_rate': Decimal('4.00'), 'open_date': datetime.date(2023, 5, 10), 'close_date': None, 'status': 'Active', 'is_joint_account': False}


In [31]:
q15 = run(db, "Q15: Customers living in a city with more than 3 customers (IN)", """
    SELECT *
    FROM customers
    WHERE city IN (
        SELECT city FROM customers GROUP BY city HAVING COUNT(*) > 3
    )
    ORDER BY city;
""")


Q15: Customers living in a city with more than 3 customers (IN)  (200 row(s))
   {'customer_id': 199, 'first_name': 'Sunita', 'last_name': 'Sharma', 'gender': 'Female', 'dob': datetime.date(2006, 3, 24), 'city': 'Bhaktapur', 'state': 'Bagmati', 'country': 'Nepal', 'phone': '9829901204', 'email': 'sunita.sharma199@mailbank.com', 'occupation': 'Farmer', 'annual_income': Decimal('3412660.60'), 'credit_score': 566, 'kyc_status': 'Pending', 'join_date': datetime.date(2016, 6, 13), 'risk_category': 'Medium'}
   {'customer_id': 133, 'first_name': 'Sarita', 'last_name': 'Tamang', 'gender': 'Male', 'dob': datetime.date(1958, 2, 2), 'city': 'Bhaktapur', 'state': 'Bagmati', 'country': 'Nepal', 'phone': '9815869993', 'email': 'sarita.tamang133@mailbank.com', 'occupation': 'Accountant', 'annual_income': Decimal('3200734.23'), 'credit_score': 594, 'kyc_status': 'Pending', 'join_date': datetime.date(2016, 3, 29), 'risk_category': 'Low'}
   {'customer_id': 175, 'first_name': 'Nisha', 'last_name': 'Sh

In [32]:
q16 = run(db, "Q16: Branches with >5 accounts, showing count + avg balance (inline view)", """
    SELECT branch, num_accounts, avg_balance
    FROM (
        SELECT branch, COUNT(*) AS num_accounts, AVG(balance) AS avg_balance
        FROM accounts
        GROUP BY branch
    ) branch_stats
    WHERE num_accounts > 5
    ORDER BY num_accounts DESC;
""")


Q16: Branches with >5 accounts, showing count + avg balance (inline view)  (8 row(s))
   {'branch': 'Pokhara City', 'num_accounts': 41, 'avg_balance': Decimal('260489.609268292683')}
   {'branch': 'Itahari Plaza', 'num_accounts': 39, 'avg_balance': Decimal('271143.368205128205')}
   {'branch': 'Bhaktapur Central', 'num_accounts': 38, 'avg_balance': Decimal('225166.650789473684')}
   {'branch': 'Kathmandu Main', 'num_accounts': 36, 'avg_balance': Decimal('188075.574722222222')}
   {'branch': 'Biratnagar East', 'num_accounts': 36, 'avg_balance': Decimal('245993.220000000000')}
   ... 3 more row(s)


## 10. Questions 17-20 — Set operations

In [33]:
q17 = run(db, "Q17: Customer ids with Savings or Checking, de-duplicated (UNION)", """
    SELECT customer_id FROM accounts WHERE account_type = 'Savings'
    UNION
    SELECT customer_id FROM accounts WHERE account_type = 'Checking';
""")


Q17: Customer ids with Savings or Checking, de-duplicated (UNION)  (105 row(s))
   {'customer_id': 184}
   {'customer_id': 116}
   {'customer_id': 71}
   {'customer_id': 68}
   {'customer_id': 52}
   ... 100 more row(s)


In [34]:
q18 = run(db, "Q18: Same list, duplicates kept (UNION ALL)", """
    SELECT customer_id FROM accounts WHERE account_type = 'Savings'
    UNION ALL
    SELECT customer_id FROM accounts WHERE account_type = 'Checking';
""")


Q18: Same list, duplicates kept (UNION ALL)  (125 row(s))
   {'customer_id': 3}
   {'customer_id': 4}
   {'customer_id': 6}
   {'customer_id': 6}
   {'customer_id': 12}
   ... 120 more row(s)


In [35]:
q19 = run(db, "Q19: Customer ids with BOTH Savings and Checking (INTERSECT)", """
    SELECT customer_id FROM accounts WHERE account_type = 'Savings'
    INTERSECT
    SELECT customer_id FROM accounts WHERE account_type = 'Checking';
""")


Q19: Customer ids with BOTH Savings and Checking (INTERSECT)  (9 row(s))
   {'customer_id': 184}
   {'customer_id': 119}
   {'customer_id': 107}
   {'customer_id': 25}
   {'customer_id': 31}
   ... 4 more row(s)


In [36]:
q20 = run(db, "Q20: Savings customers with NO Fixed Deposit account (EXCEPT)", """
    SELECT customer_id FROM accounts WHERE account_type = 'Savings'
    EXCEPT
    SELECT customer_id FROM accounts WHERE account_type = 'Fixed Deposit';
""")


Q20: Savings customers with NO Fixed Deposit account (EXCEPT)  (43 row(s))
   {'customer_id': 184}
   {'customer_id': 99}
   {'customer_id': 189}
   {'customer_id': 71}
   {'customer_id': 68}
   ... 38 more row(s)


## 11. Questions 21-25 — CTEs / Views

In [37]:
q21 = run(db, "Q21: Accounts whose total transaction amount exceeds 100,000 (CTE)", """
    WITH account_totals AS (
        SELECT account_id, SUM(amount) AS total_amount
        FROM transactions
        GROUP BY account_id
    )
    SELECT * FROM account_totals WHERE total_amount > 100000 ORDER BY total_amount DESC;
""")


Q21: Accounts whose total transaction amount exceeds 100,000 (CTE)  (257 row(s))
   {'account_id': 100178, 'total_amount': Decimal('1175970.54')}
   {'account_id': 100204, 'total_amount': Decimal('1115548.05')}
   {'account_id': 100196, 'total_amount': Decimal('1084446.45')}
   {'account_id': 100136, 'total_amount': Decimal('1072806.94')}
   {'account_id': 100206, 'total_amount': Decimal('1062740.21')}
   ... 252 more row(s)


In [38]:
q22 = run(db, "Q22: Single highest-balance account in each branch (CTE)", """
    WITH ranked AS (
        SELECT *, RANK() OVER (PARTITION BY branch ORDER BY balance DESC) AS rnk
        FROM accounts
    )
    SELECT branch, account_id, balance FROM ranked WHERE rnk = 1 ORDER BY branch;
""")


Q22: Single highest-balance account in each branch (CTE)  (8 row(s))
   {'branch': 'Bhaktapur Central', 'account_id': 100202, 'balance': Decimal('835277.30')}
   {'branch': 'Biratnagar East', 'account_id': 100172, 'balance': Decimal('838229.39')}
   {'branch': 'Butwal West', 'account_id': 100151, 'balance': Decimal('884143.08')}
   {'branch': 'Dharan North', 'account_id': 100169, 'balance': Decimal('895536.41')}
   {'branch': 'Itahari Plaza', 'account_id': 100161, 'balance': Decimal('874917.08')}
   ... 3 more row(s)


In [39]:
q23 = run(db, "Q23: Accounts whose total deposits exceed current balance (chained CTEs)", """
    WITH deposit_totals AS (
        SELECT account_id, SUM(amount) AS total_deposits
        FROM transactions
        WHERE txn_type = 'Deposit'
        GROUP BY account_id
    ),
    compare AS (
        SELECT a.account_id, a.balance, d.total_deposits
        FROM accounts a
        JOIN deposit_totals d ON a.account_id = d.account_id
    )
    SELECT * FROM compare WHERE total_deposits > balance ORDER BY total_deposits DESC;
""")


Q23: Accounts whose total deposits exceed current balance (chained CTEs)  (80 row(s))
   {'account_id': 100178, 'balance': Decimal('2854.12'), 'total_deposits': Decimal('435864.06')}
   {'account_id': 100141, 'balance': Decimal('357151.66'), 'total_deposits': Decimal('405533.53')}
   {'account_id': 100134, 'balance': Decimal('192924.43'), 'total_deposits': Decimal('325530.81')}
   {'account_id': 100108, 'balance': Decimal('68019.02'), 'total_deposits': Decimal('308843.62')}
   {'account_id': 100150, 'balance': Decimal('53296.54'), 'total_deposits': Decimal('291910.96')}
   ... 75 more row(s)


In [40]:
run(db, "Q24: CREATE VIEW active_accounts_view", """
    CREATE OR REPLACE VIEW active_accounts_view AS
    SELECT a.*, c.first_name || ' ' || c.last_name AS full_name
    FROM accounts a
    JOIN customers c ON a.customer_id = c.customer_id
    WHERE a.status = 'Active';
""")
q24 = run(db, "  -> preview of active_accounts_view", "SELECT * FROM active_accounts_view;")


Q24: CREATE VIEW active_accounts_view  (0 row(s))

  -> preview of active_accounts_view  (192 row(s))
   {'account_id': 100001, 'customer_id': 1, 'account_type': 'Checking', 'branch': 'Dharan North', 'ifsc_code': 'DHRN0006', 'currency': 'NPR', 'balance': Decimal('179640.10'), 'interest_rate': Decimal('0.00'), 'open_date': datetime.date(2023, 8, 29), 'close_date': None, 'status': 'Active', 'is_joint_account': False, 'full_name': 'Krishna Rai'}
   {'account_id': 100002, 'customer_id': 2, 'account_type': 'Fixed Deposit', 'branch': 'Bhaktapur Central', 'ifsc_code': 'BHKT0007', 'currency': 'NPR', 'balance': Decimal('757474.71'), 'interest_rate': Decimal('10.90'), 'open_date': datetime.date(2020, 10, 28), 'close_date': None, 'status': 'Active', 'is_joint_account': False, 'full_name': 'Anita Lama'}
   {'account_id': 100004, 'customer_id': 3, 'account_type': 'Fixed Deposit', 'branch': 'Lalitpur', 'ifsc_code': 'LALI0003', 'currency': 'NPR', 'balance': Decimal('84465.72'), 'interest_rate': Deci

In [41]:
run(db, "Q25a: CREATE MATERIALIZED VIEW branch_balance_summary", """
    DROP MATERIALIZED VIEW IF EXISTS branch_balance_summary;
    CREATE MATERIALIZED VIEW branch_balance_summary AS
    SELECT branch, SUM(balance) AS total_balance, COUNT(*) AS account_count
    FROM accounts
    GROUP BY branch;
""")
# CONCURRENTLY needs a unique index to work.
run(db, "Q25b: unique index required for CONCURRENTLY refresh", """
    CREATE UNIQUE INDEX IF NOT EXISTS idx_branch_balance_summary_branch
    ON branch_balance_summary (branch);
""")
run(db, "Q25c: REFRESH MATERIALIZED VIEW CONCURRENTLY branch_balance_summary", """
    REFRESH MATERIALIZED VIEW CONCURRENTLY branch_balance_summary;
""")
q25 = run(db, "  -> preview of branch_balance_summary", "SELECT * FROM branch_balance_summary ORDER BY total_balance DESC;")


Q25a: CREATE MATERIALIZED VIEW branch_balance_summary  (0 row(s))

Q25b: unique index required for CONCURRENTLY refresh  (0 row(s))

Q25c: REFRESH MATERIALIZED VIEW CONCURRENTLY branch_balance_summary  (0 row(s))

  -> preview of branch_balance_summary  (8 row(s))
   {'branch': 'Pokhara City', 'total_balance': Decimal('10680073.98'), 'account_count': 41}
   {'branch': 'Butwal West', 'total_balance': Decimal('10638192.58'), 'account_count': 35}
   {'branch': 'Itahari Plaza', 'total_balance': Decimal('10574591.36'), 'account_count': 39}
   {'branch': 'Biratnagar East', 'total_balance': Decimal('8855755.92'), 'account_count': 36}
   {'branch': 'Bhaktapur Central', 'total_balance': Decimal('8556332.73'), 'account_count': 38}
   ... 3 more row(s)


## 12. Questions 26-31 — Window functions

In [42]:
q26 = run(db, "Q26: Most recent transaction per account (ROW_NUMBER)", """
    WITH ranked AS (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY account_id ORDER BY txn_date DESC, txn_time DESC
        ) AS rn
        FROM transactions
    )
    SELECT transaction_id, account_id, txn_date, txn_time, amount
    FROM ranked WHERE rn = 1;
""")


Q26: Most recent transaction per account (ROW_NUMBER)  (279 row(s))
   {'transaction_id': 7000009, 'account_id': 100001, 'txn_date': datetime.date(2026, 7, 18), 'txn_time': datetime.time(7, 45), 'amount': Decimal('63454.54')}
   {'transaction_id': 7000035, 'account_id': 100002, 'txn_date': datetime.date(2026, 8, 28), 'txn_time': datetime.time(11, 30), 'amount': Decimal('11622.85')}
   {'transaction_id': 7000055, 'account_id': 100003, 'txn_date': datetime.date(2026, 8, 17), 'txn_time': datetime.time(21, 15), 'amount': Decimal('1260.74')}
   {'transaction_id': 7000059, 'account_id': 100004, 'txn_date': datetime.date(2026, 8, 24), 'txn_time': datetime.time(13, 15), 'amount': Decimal('41139.79')}
   {'transaction_id': 7000079, 'account_id': 100005, 'txn_date': datetime.date(2026, 6, 28), 'txn_time': datetime.time(10, 30), 'amount': Decimal('876.75')}
   ... 274 more row(s)


In [43]:
q27 = run(db, "Q27: Rank customers by total account balance (RANK, gaps allowed)", """
    SELECT customer_id, SUM(balance) AS total_balance,
           RANK() OVER (ORDER BY SUM(balance) DESC) AS balance_rank
    FROM accounts
    GROUP BY customer_id
    ORDER BY balance_rank;
""")


Q27: Rank customers by total account balance (RANK, gaps allowed)  (201 row(s))
   {'customer_id': 115, 'total_balance': Decimal('1738508.08'), 'balance_rank': 1}
   {'customer_id': 148, 'total_balance': Decimal('1562901.71'), 'balance_rank': 2}
   {'customer_id': 26, 'total_balance': Decimal('1562656.70'), 'balance_rank': 3}
   {'customer_id': 165, 'total_balance': Decimal('1474065.84'), 'balance_rank': 4}
   {'customer_id': 163, 'total_balance': Decimal('1273404.68'), 'balance_rank': 5}
   ... 196 more row(s)


In [44]:
q28 = run(db, "Q28: Rank branches by total transaction amount, no gaps (DENSE_RANK)", """
    SELECT a.branch, SUM(t.amount) AS total_amount,
           DENSE_RANK() OVER (ORDER BY SUM(t.amount) DESC) AS branch_rank
    FROM transactions t
    JOIN accounts a ON t.account_id = a.account_id
    GROUP BY a.branch
    ORDER BY branch_rank;
""")


Q28: Rank branches by total transaction amount, no gaps (DENSE_RANK)  (8 row(s))
   {'branch': 'Biratnagar East', 'total_amount': Decimal('20642705.23'), 'branch_rank': 1}
   {'branch': 'Pokhara City', 'total_amount': Decimal('18993640.07'), 'branch_rank': 2}
   {'branch': 'Butwal West', 'total_amount': Decimal('18427855.22'), 'branch_rank': 3}
   {'branch': 'Bhaktapur Central', 'total_amount': Decimal('17946807.74'), 'branch_rank': 4}
   {'branch': 'Itahari Plaza', 'total_amount': Decimal('17044322.09'), 'branch_rank': 5}
   ... 3 more row(s)


In [45]:
q29 = run(db, "Q29: Each transaction next to the previous one on the same account (LAG)", """
    SELECT transaction_id, account_id, txn_date, amount,
           LAG(amount) OVER (PARTITION BY account_id ORDER BY txn_date, txn_time) AS prev_amount
    FROM transactions
    ORDER BY account_id, txn_date, txn_time;
""")


Q29: Each transaction next to the previous one on the same account (LAG)  (4613 row(s))
   {'transaction_id': 7000011, 'account_id': 100001, 'txn_date': datetime.date(2024, 2, 12), 'amount': Decimal('272.45'), 'prev_amount': None}
   {'transaction_id': 7000017, 'account_id': 100001, 'txn_date': datetime.date(2024, 5, 5), 'amount': Decimal('46086.82'), 'prev_amount': Decimal('272.45')}
   {'transaction_id': 7000012, 'account_id': 100001, 'txn_date': datetime.date(2024, 7, 10), 'amount': Decimal('9482.80'), 'prev_amount': Decimal('46086.82')}
   {'transaction_id': 7000002, 'account_id': 100001, 'txn_date': datetime.date(2024, 9, 6), 'amount': Decimal('19581.27'), 'prev_amount': Decimal('9482.80')}
   {'transaction_id': 7000005, 'account_id': 100001, 'txn_date': datetime.date(2025, 2, 13), 'amount': Decimal('62000.39'), 'prev_amount': Decimal('19581.27')}
   ... 4608 more row(s)


In [46]:
q30 = run(db, "Q30: Each transaction next to the next one + difference (LEAD)", """
    SELECT transaction_id, account_id, txn_date, amount,
           LEAD(amount) OVER (PARTITION BY account_id ORDER BY txn_date, txn_time) AS next_amount,
           LEAD(amount) OVER (PARTITION BY account_id ORDER BY txn_date, txn_time) - amount AS diff
    FROM transactions
    ORDER BY account_id, txn_date, txn_time;
""")


Q30: Each transaction next to the next one + difference (LEAD)  (4613 row(s))
   {'transaction_id': 7000011, 'account_id': 100001, 'txn_date': datetime.date(2024, 2, 12), 'amount': Decimal('272.45'), 'next_amount': Decimal('46086.82'), 'diff': Decimal('45814.37')}
   {'transaction_id': 7000017, 'account_id': 100001, 'txn_date': datetime.date(2024, 5, 5), 'amount': Decimal('46086.82'), 'next_amount': Decimal('9482.80'), 'diff': Decimal('-36604.02')}
   {'transaction_id': 7000012, 'account_id': 100001, 'txn_date': datetime.date(2024, 7, 10), 'amount': Decimal('9482.80'), 'next_amount': Decimal('19581.27'), 'diff': Decimal('10098.47')}
   {'transaction_id': 7000002, 'account_id': 100001, 'txn_date': datetime.date(2024, 9, 6), 'amount': Decimal('19581.27'), 'next_amount': Decimal('62000.39'), 'diff': Decimal('42419.12')}
   {'transaction_id': 7000005, 'account_id': 100001, 'txn_date': datetime.date(2025, 2, 13), 'amount': Decimal('62000.39'), 'next_amount': Decimal('22994.74'), 'diff': De

In [47]:
q31 = run(db, "Q31: Running total of transactions per account, in date order", """
    SELECT transaction_id, account_id, txn_date, amount,
           SUM(amount) OVER (
               PARTITION BY account_id ORDER BY txn_date, txn_time
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
           ) AS running_total
    FROM transactions
    ORDER BY account_id, txn_date, txn_time;
""")


Q31: Running total of transactions per account, in date order  (4613 row(s))
   {'transaction_id': 7000011, 'account_id': 100001, 'txn_date': datetime.date(2024, 2, 12), 'amount': Decimal('272.45'), 'running_total': Decimal('272.45')}
   {'transaction_id': 7000017, 'account_id': 100001, 'txn_date': datetime.date(2024, 5, 5), 'amount': Decimal('46086.82'), 'running_total': Decimal('46359.27')}
   {'transaction_id': 7000012, 'account_id': 100001, 'txn_date': datetime.date(2024, 7, 10), 'amount': Decimal('9482.80'), 'running_total': Decimal('55842.07')}
   {'transaction_id': 7000002, 'account_id': 100001, 'txn_date': datetime.date(2024, 9, 6), 'amount': Decimal('19581.27'), 'running_total': Decimal('75423.34')}
   {'transaction_id': 7000005, 'account_id': 100001, 'txn_date': datetime.date(2025, 2, 13), 'amount': Decimal('62000.39'), 'running_total': Decimal('137423.73')}
   ... 4608 more row(s)


## 13. Questions 32-33 — Data quality

In [48]:
q32 = run(db, "Q32: Duplicate customers (same first_name, last_name, dob)", """
    SELECT first_name, last_name, dob, COUNT(*) AS cnt, array_agg(customer_id) AS customer_ids
    FROM customers
    GROUP BY first_name, last_name, dob
    HAVING COUNT(*) > 1;
""")


Q32: Duplicate customers (same first_name, last_name, dob)  (1 row(s))
   {'first_name': 'Sabina', 'last_name': 'Bhattarai', 'dob': datetime.date(1976, 11, 24), 'cnt': 2, 'customer_ids': [6, 9999]}


In [49]:
q33a = run(db, "Q33a: Customers missing city or email", """
    SELECT customer_id, first_name, last_name, city, email
    FROM customers
    WHERE city IS NULL OR city = '' OR email IS NULL OR email = '';
""")

q33b = run(db, "Q33b: Orphaned accounts (customer_id with no matching customer)", """
    SELECT a.*
    FROM accounts a
    LEFT JOIN customers c ON a.customer_id = c.customer_id
    WHERE c.customer_id IS NULL;
""")


Q33a: Customers missing city or email  (2 row(s))
   {'customer_id': 11, 'first_name': 'Kavita', 'last_name': 'KC', 'city': None, 'email': 'kavita.kc11@mailbank.com'}
   {'customer_id': 21, 'first_name': 'Meera', 'last_name': 'Khadka', 'city': 'Dhangadhi', 'email': None}

Q33b: Orphaned accounts (customer_id with no matching customer)  (1 row(s))
   {'account_id': 100280, 'customer_id': 99999, 'account_type': 'Savings', 'branch': 'Kathmandu Main', 'ifsc_code': 'KTMN0001', 'currency': 'NPR', 'balance': Decimal('15000.00'), 'interest_rate': Decimal('4.00'), 'open_date': datetime.date(2023, 5, 10), 'close_date': None, 'status': 'Active', 'is_joint_account': False}


## 14. Question 34 — CASE

In [50]:
q34 = run(db, "Q34: Active accounts bucketed Low/Medium/High by balance (CASE)", """
    SELECT
        CASE
            WHEN balance < 10000 THEN 'Low'
            WHEN balance BETWEEN 10000 AND 100000 THEN 'Medium'
            ELSE 'High'
        END AS balance_bucket,
        COUNT(*) AS account_count
    FROM accounts
    WHERE status = 'Active'
    GROUP BY balance_bucket
    ORDER BY balance_bucket;
""")


Q34: Active accounts bucketed Low/Medium/High by balance (CASE)  (3 row(s))
   {'balance_bucket': 'High', 'account_count': 136}
   {'balance_bucket': 'Low', 'account_count': 10}
   {'balance_bucket': 'Medium', 'account_count': 47}


## 15. Question 35 — Transaction block (UPDATE + INSERT, rollback-safe)

Deducts a 500 fee from every account with `balance > 200,000` and inserts a
matching `'Fee'` transaction row for each, inside one all-or-nothing
transaction. Any failure rolls the whole thing back. Accounts are snapshotted
**before** the `UPDATE`, since `balance - 500` is only meaningful pre-update.

In [51]:
try:
    with db.cursor() as cur:
        # 1. Snapshot affected accounts BEFORE mutating anything.
        cur.execute("""
            SELECT account_id, currency, balance - 500 AS new_balance
            FROM accounts
            WHERE balance > 200000;
        """)
        affected = cur.fetchall()

        cur.execute("SELECT COALESCE(MAX(transaction_id), 0) AS max_id FROM transactions;")
        next_id = cur.fetchone()["max_id"] + 1

        # 2. Deduct the fee. Always filter UPDATE/DELETE with WHERE.
        cur.execute("""
            UPDATE accounts SET balance = balance - 500
            WHERE balance > 200000;
        """)

        # 3. Insert one 'Fee' transaction per affected account.
        for i, row in enumerate(affected):
            cur.execute("""
                INSERT INTO transactions
                    (transaction_id, account_id, txn_date, txn_type, channel,
                     amount, currency, balance_after, description, is_flagged)
                VALUES (%s, %s, CURRENT_DATE, 'Fee', 'System',
                        %s, %s, %s, 'Monthly maintenance fee', false);
            """, (next_id + i, row["account_id"], 500, row["currency"], row["new_balance"]))

    db.commit()
    logger.info("Q35: applied 500 fee to %d accounts, inserted %d Fee transactions", len(affected), len(affected))
    print(f"Q35: fee applied to {len(affected)} account(s); {len(affected)} Fee transaction(s) inserted and committed.")
except Exception as e:
    db.rollback()
    logger.error("Q35 failed, rolled back: %s", e)
    print(f"Q35 failed and was rolled back: {e}")
    raise

Q35: fee applied to 121 account(s); 121 Fee transaction(s) inserted and committed.


## 16. Questions 36-40 — NTILE, compliance checks, correlated subquery

In [52]:
q36 = run(db, "Q36: Customers split into 4 income quartiles (NTILE)", """
    WITH quartiles AS (
        SELECT customer_id, annual_income,
               NTILE(4) OVER (ORDER BY annual_income) AS quartile
        FROM customers
    )
    SELECT quartile, COUNT(*) AS customer_count
    FROM quartiles
    GROUP BY quartile
    ORDER BY quartile;
""")


Q36: Customers split into 4 income quartiles (NTILE)  (4 row(s))
   {'quartile': 1, 'customer_count': 51}
   {'quartile': 2, 'customer_count': 50}
   {'quartile': 3, 'customer_count': 50}
   {'quartile': 4, 'customer_count': 50}


In [53]:
q37 = run(db, "Q37: credit_score < 500 but holds an account balance > 200,000", """
    SELECT DISTINCT c.customer_id, c.first_name, c.last_name, c.credit_score
    FROM customers c
    JOIN accounts a ON c.customer_id = a.customer_id
    WHERE c.credit_score < 500 AND a.balance > 200000;
""")


Q37: credit_score < 500 but holds an account balance > 200,000  (35 row(s))
   {'customer_id': 143, 'first_name': 'Radha', 'last_name': 'Maharjan', 'credit_score': 458}
   {'customer_id': 171, 'first_name': 'Hari', 'last_name': 'Acharya', 'credit_score': 403}
   {'customer_id': 135, 'first_name': 'Indira', 'last_name': 'Subedi', 'credit_score': 331}
   {'customer_id': 182, 'first_name': 'Manoj', 'last_name': 'Bhandari', 'credit_score': 343}
   {'customer_id': 164, 'first_name': 'Nisha', 'last_name': 'Lama', 'credit_score': 321}
   ... 30 more row(s)


In [54]:
q38 = run(db, "Q38: Flagged transactions with customer/branch/channel, amount desc", """
    SELECT t.transaction_id, t.amount,
           c.first_name || ' ' || c.last_name AS customer_name,
           a.branch, t.channel
    FROM transactions t
    JOIN accounts a ON t.account_id = a.account_id
    JOIN customers c ON a.customer_id = c.customer_id
    WHERE t.is_flagged = true
    ORDER BY t.amount DESC;
""")


Q38: Flagged transactions with customer/branch/channel, amount desc  (165 row(s))
   {'transaction_id': 7000679, 'amount': Decimal('94983.27'), 'customer_name': 'Gita Bhattarai', 'branch': 'Bhaktapur Central', 'channel': 'POS'}
   {'transaction_id': 7001864, 'amount': Decimal('94872.56'), 'customer_name': 'Nabin Adhikari', 'branch': 'Pokhara City', 'channel': 'Mobile App'}
   {'transaction_id': 7002380, 'amount': Decimal('94816.61'), 'customer_name': 'Ganesh Joshi', 'branch': 'Bhaktapur Central', 'channel': 'ATM'}
   {'transaction_id': 7003439, 'amount': Decimal('94685.37'), 'customer_name': 'Bipin Ghimire', 'branch': 'Butwal West', 'channel': 'Internet Banking'}
   {'transaction_id': 7002405, 'amount': Decimal('94588.07'), 'customer_name': 'Santosh Karki', 'branch': 'Lalitpur', 'channel': 'Mobile App'}
   ... 160 more row(s)


In [55]:
q39 = run(db, "Q39: kyc_status = Expired but still has an Active account (compliance risk)", """
    SELECT DISTINCT c.customer_id, c.first_name, c.last_name, c.kyc_status
    FROM customers c
    JOIN accounts a ON c.customer_id = a.customer_id
    WHERE c.kyc_status = 'Expired' AND a.status = 'Active';
""")


Q39: kyc_status = Expired but still has an Active account (compliance risk)  (26 row(s))
   {'customer_id': 66, 'first_name': 'Sabina', 'last_name': 'Joshi', 'kyc_status': 'Expired'}
   {'customer_id': 142, 'first_name': 'Sunita', 'last_name': 'Malla', 'kyc_status': 'Expired'}
   {'customer_id': 149, 'first_name': 'Naveen', 'last_name': 'Magar', 'kyc_status': 'Expired'}
   {'customer_id': 13, 'first_name': 'Bidya', 'last_name': 'Pandey', 'kyc_status': 'Expired'}
   {'customer_id': 1, 'first_name': 'Krishna', 'last_name': 'Rai', 'kyc_status': 'Expired'}
   ... 21 more row(s)


In [56]:
q40 = run(db, "Q40: Joint accounts above their branch's average balance (correlated)", """
    SELECT a.account_id, a.branch, a.balance
    FROM accounts a
    WHERE a.is_joint_account = true
      AND a.balance > (
          SELECT AVG(a2.balance) FROM accounts a2 WHERE a2.branch = a.branch
      )
    ORDER BY a.branch;
""")


Q40: Joint accounts above their branch's average balance (correlated)  (31 row(s))
   {'account_id': 100036, 'branch': 'Bhaktapur Central', 'balance': Decimal('771954.62')}
   {'account_id': 100220, 'branch': 'Bhaktapur Central', 'balance': Decimal('297664.11')}
   {'account_id': 100184, 'branch': 'Bhaktapur Central', 'balance': Decimal('471609.77')}
   {'account_id': 100200, 'branch': 'Biratnagar East', 'balance': Decimal('710413.82')}
   {'account_id': 100140, 'branch': 'Biratnagar East', 'balance': Decimal('268656.28')}
   ... 26 more row(s)


## 17. Close connection

In [57]:
db.close()
print("Connection closed. See log.log for the full operation history.")

Connection closed. See log.log for the full operation history.
